# Fine-tune the resume-fit classifier (Assignment requirement 8)

**Runtime -> Change runtime type -> T4 GPU** before running anything, then *Run all*.
Two training arms, roughly 20 minutes total on a T4 versus ~8 hours on a laptop CPU.

Works in the Colab browser and through the VS Code Colab extension alike: no
`files.upload()` / `files.download()` widgets are used, since those only
function in a real browser session.

> Cells 3 and 4 are generated from `scripts/finetune_classifier.py` and
> `app/selection.py`. Do not edit them here — edit the real files and re-run
> `python scripts/build_colab_notebook.py`.


In [11]:
# 1. Confirm a GPU is attached. If this errors, fix the runtime type first.
!nvidia-smi --query-gpu=name,memory.total --format=csv


name, memory.total [MiB]
Tesla T4, 15360 MiB


In [12]:
# 2. Dependencies. transformers is pinned to the local serving version so the
#    saved model loads there without a version surprise.
!pip install -q "transformers==4.44.2" datasets accelerate scikit-learn


In [13]:
# 3. The training script goes in scripts/ deliberately: it derives its default
#    output path from Path(__file__).parent.parent, which at /content would
#    resolve to / and save the model outside the working directory.
import os

os.makedirs('scripts', exist_ok=True)


In [14]:
%%writefile scripts/finetune_classifier.py
"""
Assignment requirement 8 — fine-tune a model on a domain dataset.

Dataset : cnamuangtoun/resume-job-description-fit  (HuggingFace Hub)
          6,241 train rows + a held-out test split
          columns: resume_text | job_description_text | label
          labels : No Fit (3143) · Potential Fit (1556) · Good Fit (1542)
Base    : distilbert-base-uncased (66M params — trains on a Colab T4 in
          minutes and runs on CPU at inference, which is the deploy target)

The dataset is imbalanced ~50/25/25, so:
  * the loss is class-weighted, otherwise the model collapses onto "No Fit"
    and reports a flattering 50% accuracy while having learned nothing;
  * the headline metric is macro-F1, not accuracy.

Run locally:
    python scripts/finetune_classifier.py --epochs 3

Run on Colab (GPU):
    !pip install -q transformers datasets accelerate scikit-learn
    !python scripts/finetune_classifier.py --epochs 3 --batch-size 32 --fp16
    # then zip models/finetuned-fit-classifier and download it
"""

import argparse
import json
from pathlib import Path

import numpy as np

PROJECT_ROOT = Path(__file__).resolve().parent.parent
DEFAULT_OUTPUT = PROJECT_ROOT / "models" / "finetuned-fit-classifier"
LABELS = ["No Fit", "Potential Fit", "Good Fit"]


def parse_args():
    parser = argparse.ArgumentParser(description="Fine-tune DistilBERT for resume-JD fit classification")
    parser.add_argument("--dataset", default="cnamuangtoun/resume-job-description-fit")
    parser.add_argument("--base-model", default="distilbert-base-uncased")
    parser.add_argument("--output-dir", default=str(DEFAULT_OUTPUT))
    parser.add_argument("--epochs", type=float, default=3.0)
    parser.add_argument("--batch-size", type=int, default=16)
    parser.add_argument("--lr", type=float, default=2e-5)
    parser.add_argument("--max-length", type=int, default=384)
    parser.add_argument("--max-train-rows", type=int, default=0, help="0 = use all rows")
    # On CPU an eval pass over the full test split costs ~13 min, and it runs
    # once per epoch plus twice more. Subsampling it is the difference between
    # a 2-hour local run and a 4-hour one. Always 0 (full split) on GPU.
    parser.add_argument("--eval-rows", type=int, default=0, help="0 = full test split")
    # The ablation this script exists to settle: does spending the same token
    # budget on JD-relevant resume content beat spending it on the first N
    # tokens? Must match app/config.py at serve time or the model is served a
    # different input distribution than it was trained on.
    parser.add_argument("--selection", default="head", choices=["head", "jd_guided"],
                        help="head = plain truncation; jd_guided = app/selection.py")
    parser.add_argument("--fp16", action="store_true", help="GPU only")
    parser.add_argument("--seed", type=int, default=42)
    return parser.parse_args()


def compute_metrics(eval_pred):
    from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support

    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    precision, recall, _, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro", zero_division=0),
        "macro_precision": precision,
        "macro_recall": recall,
        "weighted_f1": f1_score(labels, preds, average="weighted", zero_division=0),
    }


def _load_selection():
    """Import selection whether running from the repo (app.selection) or from a
    flat Colab working directory (selection.py written next to this script)."""
    import sys
    from pathlib import Path

    # Try every layout this script legitimately runs under: the repo (app.selection),
    # beside itself (Colab writes scripts/selection.py), and the working directory.
    for candidate in (Path(__file__).resolve().parent, Path.cwd()):
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))

    try:
        from app import selection
    except ImportError:
        import selection
    return selection


def main():
    import torch
    from datasets import load_dataset
    from transformers import (
        AutoModelForSequenceClassification,
        AutoTokenizer,
        DataCollatorWithPadding,
        Trainer,
        TrainingArguments,
        set_seed,
    )

    args = parse_args()
    set_seed(args.seed)

    label2id = {label: i for i, label in enumerate(LABELS)}
    id2label = {i: label for label, i in label2id.items()}

    print(f"Loading dataset {args.dataset} ...")
    raw = load_dataset(args.dataset)
    train_split, eval_split = raw["train"], raw["test"]

    if args.max_train_rows:
        train_split = train_split.shuffle(seed=args.seed).select(
            range(min(args.max_train_rows, len(train_split)))
        )
    if args.eval_rows:
        eval_split = eval_split.shuffle(seed=args.seed).select(
            range(min(args.eval_rows, len(eval_split)))
        )

    print(f"train={len(train_split)}  eval={len(eval_split)}")

    tokenizer = AutoTokenizer.from_pretrained(args.base_model)

    selection = _load_selection() if args.selection == "jd_guided" else None

    def preprocess(batch):
        jds, resumes = batch["job_description_text"], batch["resume_text"]

        if selection is not None:
            pairs = [
                selection.prepare_pair(resume, jd, max_tokens=args.max_length, embedder=None)
                for jd, resume in zip(jds, resumes)
            ]
            jds = [p[0] for p in pairs]
            resumes = [p[1] for p in pairs]

        # JD first: it is the shorter field, so truncation trims the resume
        # tail rather than dropping the requirements we match against.
        encoded = tokenizer(jds, resumes, truncation=True, max_length=args.max_length)
        encoded["labels"] = [label2id[label] for label in batch["label"]]
        return encoded

    drop = train_split.column_names
    train_ds = train_split.map(preprocess, batched=True, remove_columns=drop)
    eval_ds = eval_split.map(preprocess, batched=True, remove_columns=drop)

    counts = np.bincount(np.array(train_ds["labels"]), minlength=len(LABELS))
    class_weights = torch.tensor(
        counts.sum() / (len(LABELS) * np.maximum(counts, 1)), dtype=torch.float
    )
    print("label counts:", dict(zip(LABELS, counts.tolist())))
    print("class weights:", [round(w, 3) for w in class_weights.tolist()])

    model = AutoModelForSequenceClassification.from_pretrained(
        args.base_model,
        num_labels=len(LABELS),
        id2label=id2label,
        label2id=label2id,
    )

    class WeightedTrainer(Trainer):
        def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
            labels = inputs.pop("labels")
            outputs = model(**inputs)
            loss = torch.nn.functional.cross_entropy(
                outputs.logits, labels, weight=class_weights.to(outputs.logits.device)
            )
            return (loss, outputs) if return_outputs else loss

    training_args = TrainingArguments(
        output_dir=str(Path(args.output_dir).parent / "training-checkpoints"),
        learning_rate=args.lr,
        per_device_train_batch_size=args.batch_size,
        per_device_eval_batch_size=args.batch_size * 2,
        num_train_epochs=args.epochs,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        save_total_limit=1,
        logging_steps=50,
        fp16=args.fp16,
        report_to=[],
        seed=args.seed,
    )

    trainer = WeightedTrainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=eval_ds,
        data_collator=DataCollatorWithPadding(tokenizer),
        compute_metrics=compute_metrics,
    )

    # Evaluating before training gives the report an honest "before" column:
    # an untrained classification head, i.e. chance-level performance.
    print("\n=== Before fine-tuning (untrained head) ===")
    baseline = trainer.evaluate()
    print(json.dumps({k: round(v, 4) for k, v in baseline.items() if isinstance(v, float)}, indent=2))

    trainer.train()

    print("\n=== After fine-tuning ===")
    final = trainer.evaluate()
    print(json.dumps({k: round(v, 4) for k, v in final.items() if isinstance(v, float)}, indent=2))

    output_dir = Path(args.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    trainer.save_model(str(output_dir))
    tokenizer.save_pretrained(str(output_dir))

    (output_dir / "training_report.json").write_text(
        json.dumps(
            {
                "dataset": args.dataset,
                "base_model": args.base_model,
                "epochs": args.epochs,
                "max_length": args.max_length,
                "selection": args.selection,
                "learning_rate": args.lr,
                "train_rows": len(train_ds),
                "eval_rows": len(eval_ds),
                "label_counts": dict(zip(LABELS, counts.tolist())),
                "before_finetuning": {k: v for k, v in baseline.items() if isinstance(v, float)},
                "after_finetuning": {k: v for k, v in final.items() if isinstance(v, float)},
                "log_history": trainer.state.log_history,
            },
            indent=2,
        ),
        encoding="utf-8",
    )

    print(f"\nSaved to {output_dir}")
    print("training_report.json holds the before/after numbers and the per-epoch curve.")


if __name__ == "__main__":
    main()


Overwriting scripts/finetune_classifier.py


In [15]:
# 4. selection.py must sit in scripts/, BESIDE the training script. Running
#    `python scripts/finetune_classifier.py` puts scripts/ on sys.path -- NOT
#    the working directory -- so a copy at /content is invisible to the import
#    and arm B dies with ModuleNotFoundError.
#
#    The SAME module runs here during training and inside the served app. If
#    the two diverge, the model is served an input distribution it never
#    trained on and degrades silently: not merely a packaging detail.
pass


In [16]:
%%writefile scripts/selection.py
"""
JD-guided extractive selection.

The problem this solves, measured on the training set:

    JD tokens       median  403
    resume tokens   median 1058
    pair median           1461 tokens
    DistilBERT ceiling      512   -> only ~34% of the pair survives

Default `longest_first` truncation keeps the *first* 34%, which on a resume is
contact details, a summary blurb and the top of the most recent job. The
requirement-bearing evidence further down is discarded.

This module keeps the *most JD-relevant* 34% instead. Same token budget,
different content. It is a preprocessing change, not an architecture change,
so it costs nothing at serve time beyond one embedding pass.

The approach is the one argued for in resume_jd_matcher_build_guide.md §5 and
§16 — compare requirements against relevant resume evidence rather than whole
documents — applied where it belongs: to what the classifier gets to read.

CRITICAL: this must run identically during fine-tuning and at inference. A
model trained on selected text and served truncated text sees a different
distribution and will silently underperform. Both paths import this module;
the Colab notebook embeds this exact file.
"""

import re
from functools import lru_cache
from typing import List, Optional, Sequence

# ~4 characters per token for English prose. Used only to convert a token
# budget into a character budget for the greedy fill; the tokenizer remains
# the final authority via its own truncation.
CHARS_PER_TOKEN = 4

DEFAULT_EMBEDDER = "sentence-transformers/all-MiniLM-L6-v2"

# Lines that are pure contact/PII noise. Dropping them before selection frees
# budget AND keeps protected characteristics out of the scored text, which is
# a fairness requirement, not just an optimisation.
_PII_LINE = re.compile(
    r"^\s*(?:"
    r"[\w.+-]+@[\w-]+\.[\w.]+"                      # bare email line
    r"|(?:\+?\d[\d\s().-]{7,}\d)"                   # bare phone line
    r"|(?:https?://|www\.)\S+"                      # bare URL line
    r"|(?:address|d\.?o\.?b\.?|date of birth|gender|nationality|marital status)\s*[:\-]"
    r")\s*$",
    re.IGNORECASE,
)


def split_units(text: str, min_chars: int = 25) -> List[str]:
    """
    Split a document into selectable units.

    Resumes are line-oriented (bullets, headings), not paragraph-oriented, so
    lines are the natural unit. Very long lines are split on sentence
    boundaries so a single wall-of-text paragraph cannot monopolise the budget.
    """
    units: List[str] = []

    for line in text.split("\n"):
        line = line.strip(" \t•●▪-–—*")
        if len(line) < min_chars or _PII_LINE.match(line):
            continue
        if len(line) <= 400:
            units.append(line)
            continue
        for sentence in re.split(r"(?<=[.!?])\s+", line):
            sentence = sentence.strip()
            if len(sentence) >= min_chars:
                units.append(sentence)

    return units


@lru_cache(maxsize=2)
def _embedder(model_name: str):
    from sentence_transformers import SentenceTransformer

    return SentenceTransformer(model_name)


def _semantic_scores(query: str, units: Sequence[str], model_name: str):
    import numpy as np

    model = _embedder(model_name)
    vectors = model.encode(
        [query, *units],
        normalize_embeddings=True,
        show_progress_bar=False,
    )
    return np.asarray(vectors[1:] @ vectors[0])


def _lexical_scores(query: str, units: Sequence[str]):
    """
    TF-IDF fallback for when no embedder is available (offline, or the Colab
    training job choosing not to pay for one). Weaker: it cannot match
    "REST API design" to "built Flask endpoints", which is exactly the kind of
    paraphrase this task is full of.
    """
    import numpy as np
    from sklearn.feature_extraction.text import TfidfVectorizer

    matrix = TfidfVectorizer(stop_words="english", sublinear_tf=True).fit_transform(
        [query, *units]
    )
    query_vector = matrix[0]
    unit_vectors = matrix[1:]
    scores = (unit_vectors @ query_vector.T).toarray().ravel()
    norms = np.sqrt(unit_vectors.multiply(unit_vectors).sum(axis=1)).A.ravel()
    denominator = norms * np.sqrt(query_vector.multiply(query_vector).sum())
    return np.divide(scores, denominator, out=np.zeros_like(scores), where=denominator > 0)


def select_relevant(
    text: str,
    query: str,
    max_tokens: int,
    embedder: Optional[str] = DEFAULT_EMBEDDER,
) -> str:
    """
    Return the subset of `text` most relevant to `query`, within `max_tokens`.

    Selected units are re-emitted in their **original document order**, not in
    score order: a resume read out of sequence loses the chronology that makes
    "5 years at X, then Y" interpretable, and the encoder sees position.

    Falls back to the head of the text if there is nothing to select from, so
    this can never return less information than plain truncation.
    """
    budget_chars = max_tokens * CHARS_PER_TOKEN
    units = split_units(text)

    if not units:
        return text[:budget_chars].strip()

    # Already fits: selection would only risk dropping evidence.
    if sum(len(u) + 1 for u in units) <= budget_chars:
        return "\n".join(units)

    try:
        scores = _semantic_scores(query, units, embedder) if embedder else _lexical_scores(query, units)
    except Exception:
        scores = _lexical_scores(query, units)

    ranked = sorted(range(len(units)), key=lambda i: float(scores[i]), reverse=True)

    chosen: List[int] = []
    used = 0
    for index in ranked:
        cost = len(units[index]) + 1
        if used + cost > budget_chars:
            continue          # keep scanning: a shorter unit may still fit
        chosen.append(index)
        used += cost

    if not chosen:            # every unit individually exceeds the budget
        return units[ranked[0]][:budget_chars].strip()

    return "\n".join(units[i] for i in sorted(chosen))


def prepare_pair(
    resume_text: str,
    job_description: str,
    max_tokens: int = 512,
    embedder: Optional[str] = DEFAULT_EMBEDDER,
) -> tuple[str, str]:
    """
    Fit a (job_description, resume) pair into `max_tokens` for a 512-token
    encoder. Returned in (jd, resume) order, matching the tokenizer call in
    the training script where the JD is passed first.

    The two sides are treated ASYMMETRICALLY, and that asymmetry is the whole
    point:

      * The job description is kept in document order and merely truncated.
        It defines the requirements. Selecting JD lines by their similarity to
        the resume would delete precisely the requirements the candidate fails
        to meet -- which is the evidence for "No Fit" -- and bias every
        prediction toward a match. An earlier version of this function did
        exactly that and measurably hurt evidence coverage.

      * The resume is filtered by relevance to the JD. Here the ranking is
        sound: a resume is a pile of evidence, most of it irrelevant to any
        one role, and the goal is to surface the part that bears on this role
        rather than whatever happens to appear in the first 254 tokens.
    """
    per_side = (max_tokens - 3) // 2      # [CLS] jd [SEP] resume [SEP]

    jd_units = split_units(job_description)
    jd_text = "\n".join(jd_units) if jd_units else job_description
    jd_kept = jd_text[: per_side * CHARS_PER_TOKEN].strip()

    resume_kept = select_relevant(resume_text, job_description, per_side, embedder)

    return jd_kept, resume_kept


Writing scripts/selection.py


In [17]:
# 5a. ARM A -- head truncation (the baseline strategy).
#
#     --max-length 512 is DistilBERT's ceiling and it matters here: the median
#     resume+JD pair is ~1,460 tokens, so 384 fed the model 26% of each pair
#     and 512 raises that to 34%. Truncation, not capacity, is this model's
#     binding constraint.
#
#     --epochs 6 because macro-F1 was still climbing at epoch 3
#     (0.248 -> 0.347 -> 0.380): the first run stopped short of converging.
!python scripts/finetune_classifier.py \
    --epochs 6 --batch-size 16 --lr 3e-5 --max-length 512 --fp16 \
    --selection head --output-dir models/fit-head


2026-08-03 15:29:16.931471: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Loading dataset cnamuangtoun/resume-job-description-fit ...
train=6241  eval=1759
/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Map: 100% 1759/1759 [00:13<00:00, 128.54 examples/s]
label counts: {'No Fit': 3143, 'Potential Fit': 1556, 'Good Fit': 1542}
class weights: [0.662, 1.337, 1.349]
Some weights of DistilBertForSequenceClassific

In [18]:
# 5b. ARM B -- JD-guided selection. Identical in every other respect, so any
#     difference is attributable to the truncation strategy alone.
#     Preprocessing costs ~11 ms/row (TF-IDF), about 90s over the dataset.
!python scripts/finetune_classifier.py \
    --epochs 6 --batch-size 16 --lr 3e-5 --max-length 512 --fp16 \
    --selection jd_guided --output-dir models/fit-jd-guided


2026-08-03 15:44:28.839068: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Loading dataset cnamuangtoun/resume-job-description-fit ...
train=6241  eval=1759
/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Map: 100% 6241/6241 [00:32<00:00, 191.50 examples/s]
Map: 100% 1759/1759 [00:08<00:00, 196.44 examples/s]
label counts: {'No Fit': 3143, 'Potential Fit': 1556, 'Good Fit': 1542}
class weights: [0.662, 1.337, 

In [19]:
# 5. The ablation table -- this is the report screenshot.
import json
import shutil
from pathlib import Path

arms = {'head truncation': 'models/fit-head',
        'JD-guided selection': 'models/fit-jd-guided'}

reports = {}
for name, directory in arms.items():
    path = Path(directory) / 'training_report.json'
    if path.exists():
        reports[name] = json.loads(path.read_text())
    else:
        print('MISSING {} -- did that training cell fail?'.format(path))

if not reports:
    raise SystemExit('No training reports found. Re-run cells 5a and 5b.')

row = '{:<22}{:>10.4f}{:>10.4f}{:>13.4f}'
print('{:<22}{:>10}{:>10}{:>13}'.format('arm', 'accuracy', 'macro_f1', 'weighted_f1'))

base = next(iter(reports.values()))['before_finetuning']
print(row.format('untrained head', base['eval_accuracy'],
                 base['eval_macro_f1'], base['eval_weighted_f1']))

for name, report in reports.items():
    after = report['after_finetuning']
    print(row.format(name, after['eval_accuracy'],
                     after['eval_macro_f1'], after['eval_weighted_f1']))

# Ship whichever arm won as the model the application serves. Deciding here,
# on the numbers, avoids a human copying the wrong directory later.
winner = max(reports, key=lambda k: reports[k]['after_finetuning']['eval_macro_f1'])
print('\nwinner: ' + winner)

shutil.rmtree('models/finetuned-fit-classifier', ignore_errors=True)
shutil.copytree(arms[winner], 'models/finetuned-fit-classifier')
Path('models/finetuned-fit-classifier/selection_strategy.txt').write_text(
    reports[winner]['selection'])
print('copied to models/finetuned-fit-classifier')


arm                     accuracy  macro_f1  weighted_f1
untrained head            0.3758    0.2960       0.3490
head truncation           0.4611    0.4004       0.4422
JD-guided selection       0.4605    0.3776       0.4289

winner: head truncation
copied to models/finetuned-fit-classifier


In [20]:
# 6. Package the winning model plus BOTH training reports, so the ablation can
#    be written up locally without needing the runtime again. The reports are
#    globbed rather than named: a failed arm then costs one missing row in the
#    table instead of aborting the zip and stranding the model on the runtime.
!zip -rq finetuned.zip models/finetuned-fit-classifier
!zip -rq finetuned.zip models/fit-*/training_report.json
!ls -lh finetuned.zip
!unzip -l finetuned.zip | grep training_report


-rw-r--r-- 1 root root 236M Aug  3 15:58 finetuned.zip
    12757  2026-08-03 15:44   models/finetuned-fit-classifier/training_report.json
    12757  2026-08-03 15:44   models/fit-head/training_report.json
    12755  2026-08-03 15:58   models/fit-jd-guided/training_report.json


## Getting `finetuned.zip` back to the project

**VS Code Colab extension:** open the Colab icon in the Activity Bar, find
`finetuned.zip` in the *Contents* view, right-click -> Download.

**Colab in a browser:** run `from google.colab import files;
files.download('finetuned.zip')` in a new cell.

Then locally:

```bash
unzip finetuned.zip -d .          # creates models/finetuned-fit-classifier/
python scripts/smoke_test.py      # classify_fit_finetuned should PASS
python scripts/evaluate.py --limit 100
```

`evaluate.py` writes `logs/eval_report.json` — the three-arm head-to-head,
also served as metric **M7** on `GET /metrics`.
